# Séance 3 — Génération de la base de données « qualité de l'air »

Ce notebook **n'est pas un TP** : c'est le script qui a servi à fabriquer la base de données que vous allez interroger dans `exercice.ipynb`. Il génère, de façon synthétique mais réaliste, les relevés de quatre réseaux de capteurs atmosphériques répartis sur la France métropolitaine, et les enregistre dans un fichier `data/qualite_air.duckdb`.

Vous n'avez **pas besoin de le lire en détail** pour faire le TP (`exercice.ipynb`) : il suffit de l'exécuter une fois (quelques secondes, aucune sortie surprenante) pour générer `data/qualite_air.duckdb` si ce fichier n'existe pas encore chez vous — il n'est pas versionné dans le dépôt (comme les autres fichiers de données volumineux du module). Le détail du code est documenté ici pour trois raisons :

- comprendre d'où viennent les données (et donc pourquoi elles ont la forme qu'elles ont — valeurs manquantes, aberrantes, colonnes inutiles, noms de colonnes différents entre tables) ;
- lire la méthodologie de chaque réseau de capteurs, utile pour juger si vos résultats sont plausibles ;
- si la curiosité vous prend, changer les paramètres en tête de notebook (nombre de capteurs, taux de bruit...) et regénérer votre propre base.

**Reproductibilité** : toute la génération aléatoire part d'une seule graine (`numpy.random.default_rng(SEED)`), donc relancer ce notebook sans rien changer produit exactement la même base.


## Installation des dépendances

Si votre environnement n'est pas encore prêt (voir le `README.md` à la racine), exécutez la cellule suivante.


In [19]:
%pip install -q -r ../../../requirements.txt



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
from pathlib import Path

import duckdb
import folium
import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.express as px
from pyproj import Transformer
from shapely.geometry import Point, Polygon

SEED = 2026
RNG = np.random.default_rng(SEED)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
CHEMIN_DB = DATA_DIR / "qualite_air.duckdb"

# Nombre de capteurs par réseau — modifiable.
N_CAPTEURS = {
    "meteo": 180,
    "qualite_air_urbain": 220,
    "ozone_rural": 90,
    "citoyen_pm": 500,
}

# Part de valeurs manquantes par réseau (sur chaque colonne mesurée) — modifiable.
TAUX_NAN = {
    "meteo": 0.01,
    "qualite_air_urbain": 0.02,
    "ozone_rural": 0.03,
    "citoyen_pm": 0.08,
}

# Part de valeurs aberrantes par réseau (glitch capteur : valeur multipliée par un facteur absurde) — modifiable.
TAUX_ABERRANT = {
    "meteo": 0.01,
    "qualite_air_urbain": 0.02,
    "ozone_rural": 0.01,
    "citoyen_pm": 0.05,
}

# Longueur caractéristique (km) de la zone d'influence d'une ville : L = L_MIN + K_POP * sqrt(population).
L_MIN_KM = 5.0
K_POP = 0.03

# Cadrage géographique commun à toutes les cartes plotly du notebook (un vrai fond de carte, pas notre
# polygone simplifié) : scope="europe" est la plus petite échelle plotly qui inclut un pays entier avec ses
# frontières réelles, on recadre ensuite sur la France avec center + projection_scale.
CONFIG_GEO_FRANCE = dict(
    scope="europe",
    center=dict(lon=2.5, lat=46.6),
    projection_scale=6,
    showland=True, landcolor="whitesmoke",
    showcountries=True, countrycolor="lightgray",
)


## 1. Les villes de référence et leur zone d'influence

On réutilise le jeu des 30 villes françaises de la séance 1. À chaque ville, on associe une **longueur caractéristique** `L_ville` (en km), qui représente la taille de son empreinte sur la pollution environnante — une grande ville pollue plus loin autour d'elle qu'un bourg. On la fait croître avec la racine de la population plutôt que la population elle-même : ça évite que Paris (10x plus peuplé que Dijon) n'ait une empreinte 10x plus grande, ce qui serait irréaliste (l'étalement urbain ne grossit pas linéairement avec la population).

Cette longueur caractéristique sert ensuite de paramètre à une fonction d'« influence » : plus on est près d'une ville (par rapport à son `L_ville`), plus son effet sur la pollution environnante est fort. C'est la même famille de formule que celle que vous utiliserez dans le TP pour interpoler spatialement une donnée à partir des capteurs (le poids gaussien `exp(-(distance / L)^2)`) — on l'utilise ici pour *construire* des relevés plausibles, vous l'utiliserez *à l'envers* pour reconstruire une carte à partir de relevés.


In [21]:
villes_data = [
    {"ville": "Paris", "region": "Île-de-France", "latitude": 48.8566, "longitude": 2.3522, "population": 2145000},
    {"ville": "Marseille", "region": "Provence-Alpes-Côte d'Azur", "latitude": 43.2965, "longitude": 5.3698, "population": 870000},
    {"ville": "Lyon", "region": "Auvergne-Rhône-Alpes", "latitude": 45.7640, "longitude": 4.8357, "population": 522000},
    {"ville": "Toulouse", "region": "Occitanie", "latitude": 43.6047, "longitude": 1.4442, "population": 493000},
    {"ville": "Nice", "region": "Provence-Alpes-Côte d'Azur", "latitude": 43.7102, "longitude": 7.2620, "population": 342000},
    {"ville": "Nantes", "region": "Pays de la Loire", "latitude": 47.2184, "longitude": -1.5536, "population": 314000},
    {"ville": "Montpellier", "region": "Occitanie", "latitude": 43.6108, "longitude": 3.8767, "population": 295000},
    {"ville": "Strasbourg", "region": "Grand Est", "latitude": 48.5734, "longitude": 7.7521, "population": 291000},
    {"ville": "Bordeaux", "region": "Nouvelle-Aquitaine", "latitude": 44.8378, "longitude": -0.5792, "population": 260000},
    {"ville": "Lille", "region": "Hauts-de-France", "latitude": 50.6292, "longitude": 3.0573, "population": 233000},
    {"ville": "Rennes", "region": "Bretagne", "latitude": 48.1173, "longitude": -1.6778, "population": 220000},
    {"ville": "Reims", "region": "Grand Est", "latitude": 49.2583, "longitude": 4.0317, "population": 182000},
    {"ville": "Le Havre", "region": "Normandie", "latitude": 49.4944, "longitude": 0.1079, "population": 170000},
    {"ville": "Saint-Étienne", "region": "Auvergne-Rhône-Alpes", "latitude": 45.4397, "longitude": 4.3872, "population": 172000},
    {"ville": "Toulon", "region": "Provence-Alpes-Côte d'Azur", "latitude": 43.1242, "longitude": 5.9280, "population": 171000},
    {"ville": "Grenoble", "region": "Auvergne-Rhône-Alpes", "latitude": 45.1885, "longitude": 5.7245, "population": 158000},
    {"ville": "Dijon", "region": "Bourgogne-Franche-Comté", "latitude": 47.3220, "longitude": 5.0415, "population": 156000},
    {"ville": "Angers", "region": "Pays de la Loire", "latitude": 47.4784, "longitude": -0.5632, "population": 152000},
    {"ville": "Nîmes", "region": "Occitanie", "latitude": 43.8367, "longitude": 4.3601, "population": 150000},
    {"ville": "Villeurbanne", "region": "Auvergne-Rhône-Alpes", "latitude": 45.7667, "longitude": 4.8794, "population": 149000},
    {"ville": "Clermont-Ferrand", "region": "Auvergne-Rhône-Alpes", "latitude": 45.7772, "longitude": 3.0870, "population": 147000},
    {"ville": "Le Mans", "region": "Pays de la Loire", "latitude": 48.0061, "longitude": 0.1996, "population": 143000},
    {"ville": "Aix-en-Provence", "region": "Provence-Alpes-Côte d'Azur", "latitude": 43.5297, "longitude": 5.4474, "population": 143000},
    {"ville": "Brest", "region": "Bretagne", "latitude": 48.3904, "longitude": -4.4861, "population": 139000},
    {"ville": "Tours", "region": "Centre-Val de Loire", "latitude": 47.3941, "longitude": 0.6848, "population": 136000},
    {"ville": "Amiens", "region": "Hauts-de-France", "latitude": 49.8941, "longitude": 2.2957, "population": 133000},
    {"ville": "Limoges", "region": "Nouvelle-Aquitaine", "latitude": 45.8336, "longitude": 1.2611, "population": 131000},
    {"ville": "Annecy", "region": "Auvergne-Rhône-Alpes", "latitude": 45.8992, "longitude": 6.1294, "population": 128000},
    {"ville": "Perpignan", "region": "Occitanie", "latitude": 42.6887, "longitude": 2.8948, "population": 119000},
    {"ville": "Metz", "region": "Grand Est", "latitude": 49.1193, "longitude": 6.1757, "population": 116000},
]

df_villes = pd.DataFrame(villes_data)
df_villes["longueur_caracteristique_km"] = L_MIN_KM + K_POP * np.sqrt(df_villes["population"])
df_villes.sort_values("population", ascending=False).head()


,ville,region,latitude,longitude,population,longueur_caracteristique_km
0,Paris,Île-de-France,48.8566,2.3522,2145000,48.937456
1,Marseille,Provence-Alpes-Côte d'Azur,43.2965,5.3698,870000,32.982137
2,Lyon,Auvergne-Rhône-Alpes,45.7640,4.8357,522000,26.674870
3,Toulouse,Occitanie,43.6047,1.4442,493000,26.064188
4,Nice,Provence-Alpes-Côte d'Azur,43.7102,7.2620,342000,22.544230


In [22]:
# Transformations de coordonnées : WGS84 (lon/lat, degrés) <-> Lambert-93 (mètres, projection officielle France).
# On calcule les distances et les zones d'influence en mètres, comme vu en séance 1 pour tout calcul métrique en France.
VERS_L93 = Transformer.from_crs(4326, 2154, always_xy=True)
VERS_WGS84 = Transformer.from_crs(2154, 4326, always_xy=True)

X_VILLES, Y_VILLES = VERS_L93.transform(df_villes["longitude"].to_numpy(), df_villes["latitude"].to_numpy())
POIDS_VILLES = (df_villes["population"] / df_villes["population"].max()).to_numpy()
L_VILLES_KM = df_villes["longueur_caracteristique_km"].to_numpy()


def influence_urbaine(lat, lon):
    """Influence cumulée (0 à 1) des 30 villes de référence au(x) point(s) (lat, lon)."""
    lat = np.atleast_1d(lat).astype(float)
    lon = np.atleast_1d(lon).astype(float)
    x, y = VERS_L93.transform(lon, lat)
    x, y = np.asarray(x), np.asarray(y)
    dist_km = np.sqrt((x[:, None] - X_VILLES[None, :]) ** 2 + (y[:, None] - Y_VILLES[None, :]) ** 2) / 1000
    contribution = POIDS_VILLES[None, :] * np.exp(-(dist_km / L_VILLES_KM[None, :]) ** 2)
    valeur = np.minimum(1.0, contribution.sum(axis=1))
    return valeur if valeur.size > 1 else valeur[0]


On définit aussi tout de suite le contour géographique de la France métropolitaine (`FRANCE_METRO_POLYGON`), qu'on utilisera pour deux choses : ne représenter le champ d'influence qu'à l'intérieur du territoire (carte ci-dessous), et, en partie 2, tirer des positions de capteurs uniquement sur le territoire français. Ce contour est **volontairement simplifié** (une quarantaine de points le long des frontières et du littoral) : largement suffisant pour une résolution de travail de 10 km, mais à ne pas réutiliser pour un usage géodésique sérieux.


In [23]:
FRANCE_METRO_POLYGON = Polygon([
    (2.37, 51.03), (3.28, 50.71), (4.00, 50.28), (4.72, 49.93), (5.78, 49.53),
    (6.90, 49.18), (7.77, 48.58), (7.45, 47.55), (6.35, 46.90), (6.87, 46.02),
    (6.68, 45.20), (6.65, 44.90), (7.56, 44.09), (7.50, 43.78), (7.27, 43.70),
    (6.65, 43.40), (5.93, 43.12), (5.37, 43.30), (3.93, 43.53), (3.03, 42.70),
    (2.87, 42.47), (1.44, 42.55), (-0.44, 42.78), (-1.77, 43.37), (-1.56, 43.48),
    (-1.25, 44.65), (-1.03, 45.63), (-1.15, 46.16), (-2.21, 47.27), (-3.13, 47.48),
    (-3.37, 47.75), (-4.78, 48.38), (-3.98, 48.72), (-1.99, 48.65), (-1.62, 49.65),
    (0.11, 49.49), (1.08, 49.93), (1.86, 50.95), (2.37, 51.03),
])
LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = FRANCE_METRO_POLYGON.bounds


La carte ci-dessous affiche ce champ d'influence (plus c'est clair, plus l'empreinte urbaine est forte), **restreint aux points situés dans le contour de la France** : on reconnaît les grandes agglomérations et leur rayon d'action respectif.


In [24]:
# 1. On garde un maillage rectangulaire simple...
lon_grille = np.linspace(LON_MIN, LON_MAX, 160)
lat_grille = np.linspace(LAT_MIN, LAT_MAX, 160)
lon_mesh, lat_mesh = np.meshgrid(lon_grille, lat_grille)

# 2. ...mais on ne garde que les points qui tombent à l'intérieur du contour de la France
#    (sinon la carte afficherait un carré coloré jusqu'en pleine Manche ou en Espagne).
candidats = gpd.GeoDataFrame(
    {"latitude": lat_mesh.ravel(), "longitude": lon_mesh.ravel()},
    geometry=gpd.points_from_xy(lon_mesh.ravel(), lat_mesh.ravel()),
)
df_champ = candidats[candidats.within(FRANCE_METRO_POLYGON)][["latitude", "longitude"]].reset_index(drop=True)

# 3. On calcule la densité (le champ d'influence) uniquement sur les points conservés.
df_champ["influence"] = influence_urbaine(df_champ["latitude"].to_numpy(), df_champ["longitude"].to_numpy())

# 4. On affiche une vraie carte de densité (lissage façon carte de chaleur) plutôt qu'une mosaïque de
#    carrés, avec un fond de carte réel — ici px.density_map, sur des tuiles "carto-positron" (les
#    mêmes que celles utilisées par folium ailleurs dans ce module), sans compte ni jeton à créer.
fig = px.density_map(
    df_champ, lat="latitude", lon="longitude", z="influence",
    radius=12, center=dict(lat=46.6, lon=2.3), zoom=4.6,
    map_style="carto-positron",
    color_continuous_scale="Inferno",
    title="Champ d'influence urbaine (mélange de gaussiennes centrées sur les villes)",
)
fig.add_scattermap(
    lat=df_villes["latitude"], lon=df_villes["longitude"],
    mode="markers+text", text=df_villes["ville"], textposition="top center",
    marker=dict(size=df_villes["population"] / 60000, color="cyan"),
    name="villes",
)
fig.update_layout(height=650, margin=dict(l=0, r=0, t=40, b=0))
fig.show()


## 2. Générer des points aléatoires en France métropolitaine

Un tirage uniforme dans le rectangle `[LON_MIN, LON_MAX] x [LAT_MIN, LAT_MAX]` placerait des capteurs en pleine Manche ou en Espagne. La solution classique est le **rejet** (*rejection sampling*) : on tire un point au hasard dans ce rectangle englobant, et on ne le garde que s'il tombe dans `FRANCE_METRO_POLYGON` (défini en partie 1).


In [25]:
def position_aleatoire_france(rng):
    """Tire un point uniformément *à l'intérieur* du contour de la France (rejection sampling)."""
    while True:
        lon, lat = rng.uniform(LON_MIN, LON_MAX), rng.uniform(LAT_MIN, LAT_MAX)
        if FRANCE_METRO_POLYGON.contains(Point(lon, lat)):
            return lon, lat


def position_biaisee_ville(index_ville, rng):
    """Tire un point autour d'une ville : un décalage gaussien en mètres (Lambert-93), rejeté si hors de France."""
    longueur_m = L_VILLES_KM[index_ville] * 1000
    while True:
        dx, dy = rng.normal(0, longueur_m, size=2)
        lon, lat = VERS_WGS84.transform(X_VILLES[index_ville] + dx, Y_VILLES[index_ville] + dy)
        if FRANCE_METRO_POLYGON.contains(Point(lon, lat)):
            return lon, lat


def tirer_positions(n, rng, biais_villes=0.0, eviter_villes_seuil=None):
    """
    Tire n positions (lon, lat).
    - biais_villes : probabilité qu'une position soit un tirage gaussien autour d'une ville (pondérée par
      population) plutôt qu'un tirage uniforme sur tout le territoire.
    - eviter_villes_seuil : si fourni, les tirages uniformes dont l'influence urbaine dépasse ce seuil sont
      rejetés (sert à placer des stations "rurales" loin des agglomérations).
    """
    poids_pop = (df_villes["population"] / df_villes["population"].sum()).to_numpy()
    positions = []
    for _ in range(n):
        if rng.random() < biais_villes:
            index_ville = rng.choice(len(df_villes), p=poids_pop)
            lon, lat = position_biaisee_ville(index_ville, rng)
        else:
            lon, lat = position_aleatoire_france(rng)
            if eviter_villes_seuil is not None:
                while influence_urbaine(lat, lon) > eviter_villes_seuil:
                    lon, lat = position_aleatoire_france(rng)
        positions.append((lon, lat))
    return positions


In [26]:
echantillon = tirer_positions(400, RNG, biais_villes=0.0)
lons_ech, lats_ech = zip(*echantillon)

fig = px.scatter_geo(
    lat=list(lats_ech), lon=list(lons_ech),
    title="Vérification : 400 points tirés uniformément dans le contour de la France",
)
fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_geos(**CONFIG_GEO_FRANCE)
fig.show()


## 3. Quatre réseaux de capteurs

Chaque réseau mesure un sous-ensemble de variables, avec sa propre densité spatiale, son propre identifiant de capteur (nom de colonne différent à chaque fois — ce sera à vous de les harmoniser), et ses propres taux de valeurs manquantes/aberrantes.

| Réseau | Table | Variables mesurées | Colonne ID | Répartition spatiale |
|---|---|---|---|---|
| Stations météo (type Météo-France) | `releves_meteo` | température, humidité, précipitations, vent (vitesse + direction) | `id_station_meteo` | Plutôt uniforme sur le territoire |
| Qualité de l'air urbain | `releves_qualite_air_urbain` | NO2, PM10, PM2.5 | `code_capteur_aq` | Très concentrée autour des villes |
| Ozone rural | `releves_ozone_rural` | O3 | `ref_station_o3` | Volontairement éloignée des centres-villes |
| Capteurs citoyens (low-cost) | `releves_citoyen_pm` | PM10 | `capteur_uid_pm` | Dense, concentrée près des villes, mais bruitée |

Chaque table contient aussi 2-3 colonnes de métadonnées **sans aucun intérêt pour calculer la qualité de l'air** (fabricant, numéro de série, pseudo du propriétaire...) — comme sur de vraies données, il faudra apprendre à les ignorer plutôt qu'à les exploiter.


In [27]:
def injecter_nan_aberrant(serie, taux_nan, taux_aberrant, rng, facteur_aberrant=9):
    """Remplace une partie des valeurs d'une colonne par des NaN, et une autre partie par des valeurs
    aberrantes (valeur multipliée par un facteur absurde, pour simuler un glitch capteur)."""
    serie = serie.copy()
    n = len(serie)
    ordre = rng.permutation(n)
    n_nan = int(n * taux_nan)
    n_aberrant = int(n * taux_aberrant)
    idx_nan = ordre[:n_nan]
    idx_aberrant = ordre[n_nan:n_nan + n_aberrant]
    serie.iloc[idx_nan] = np.nan
    serie.iloc[idx_aberrant] = serie.iloc[idx_aberrant] * facteur_aberrant
    return serie


def carte_capteurs(df, lat_col, lon_col, couleur_col, id_col, titre):
    centre = [df[lat_col].mean(), df[lon_col].mean()]
    carte = folium.Map(location=centre, zoom_start=6, tiles="cartodbpositron")
    valeurs = df[couleur_col].dropna()
    vmin, vmax = valeurs.min(), valeurs.max()
    for _, row in df.iterrows():
        valeur = row[couleur_col]
        if pd.isna(valeur):
            couleur = "gray"
        else:
            ratio = 0 if vmax == vmin else (valeur - vmin) / (vmax - vmin)
            couleur = f"#{int(255 * ratio):02x}{int(80 * (1 - ratio)):02x}{int(255 * (1 - ratio)):02x}"
        folium.CircleMarker(
            location=[row[lat_col], row[lon_col]],
            radius=4,
            color=couleur,
            fill=True,
            fill_opacity=0.8,
            popup=f"{id_col} = {row[id_col]} · {couleur_col} = {valeur}",
        ).add_to(carte)
    folium.map.Marker(centre, icon=folium.DivIcon(html=f"<div style='font-size:14pt'><b>{titre}</b></div>")).add_to(carte)
    return carte


def carte_capteurs_plotly(df, lat_col, lon_col, couleur_col, id_col, titre, colonnes_survol=None):
    """Équivalent plotly de carte_capteurs : moins riche en interactions que folium, mais s'affiche
    directement dans la sortie de la cellule (pas besoin d'ouvrir un fichier HTML à part)."""
    fig = px.scatter_geo(
        df, lat=lat_col, lon=lon_col, color=couleur_col,
        hover_name=id_col, hover_data=colonnes_survol or {},
        color_continuous_scale="YlOrRd", title=titre,
    )
    fig.update_traces(marker=dict(size=6))
    fig.update_geos(**CONFIG_GEO_FRANCE)
    fig.update_layout(height=550)
    #fig.show()
    return fig


### 3.1. Réseau météo

**Méthodologie.** Les stations météo sont réparties de façon globalement homogène sur le territoire (seul 1 capteur sur 4 est tiré avec un léger biais vers les villes, pour refléter la densification réelle des stations secondaires en zone peuplée). Les valeurs ne dépendent pas de la proximité urbaine, mais de la latitude :

- **température** : décroît avec la latitude (Nord plus froid que Sud), ordre de grandeur 5-25°C selon la position ;
- **humidité** : anti-corrélée à la température, 40-90% ;
- **précipitations journalières** : la plupart des stations sont à 0 ou presque (pas de pluie ce jour-là), avec une distribution à queue lourde (loi exponentielle) pour les jours pluvieux, 0-40 mm ;
- **vent** : vitesse 0-70 km/h (loi gamma, asymétrique — les tempêtes sont rares mais fortes), direction uniforme 0-360°.

Colonnes inutiles ajoutées : `fabricant_capteur`, `altitude_station_m` (réaliste en soi, mais non exploitée par la formule de qualité de l'air qu'on va construire), `derniere_maintenance`.


In [28]:
n = N_CAPTEURS["meteo"]
positions = tirer_positions(n, RNG, biais_villes=0.25)
lons, lats = zip(*positions)

temperature_c = 24 - (np.array(lats) - 43) * 0.9 + RNG.normal(0, 1.2, n)
pluie_aujourdhui = RNG.random(n) < 0.35
humidite_pct = np.clip(70 - (temperature_c - 15) * 1.0 + RNG.normal(0, 8, n), 20, 100)
precipitation_mm_jour = np.where(pluie_aujourdhui, RNG.exponential(scale=4, size=n), 0.0)
vent_vitesse_kmh = np.clip(RNG.gamma(shape=3, scale=5, size=n), 0, None)
vent_direction_deg = RNG.uniform(0, 360, n)

df_meteo = pd.DataFrame({
    "id_station_meteo": [f"MTO-{i:04d}" for i in range(n)],
    "latitude": lats,
    "longitude": lons,
    "altitude_station_m": RNG.integers(0, 1500, n),
    "fabricant_capteur": RNG.choice(["VaisalaTech", "MeteoSens", "ClimaWatch"], n),
    "derniere_maintenance": pd.to_datetime("2026-01-01") - pd.to_timedelta(RNG.integers(0, 700, n), unit="D"),
    "temperature_c": temperature_c.round(1),
    "humidite_pct": humidite_pct.round(1),
    "precipitation_mm_jour": precipitation_mm_jour.round(1),
    "vent_vitesse_kmh": vent_vitesse_kmh.round(1),
    "vent_direction_deg": vent_direction_deg.round(0),
})

for col in ["temperature_c", "humidite_pct", "precipitation_mm_jour", "vent_vitesse_kmh", "vent_direction_deg"]:
    df_meteo[col] = injecter_nan_aberrant(df_meteo[col], TAUX_NAN["meteo"], TAUX_ABERRANT["meteo"], RNG)

df_meteo.head()


,id_station_meteo,latitude,longitude,altitude_station_m,fabricant_capteur,derniere_maintenance,temperature_c,humidite_pct,precipitation_mm_jour,vent_vitesse_kmh,vent_direction_deg
0,MTO-0000,45.249830,3.982938,1202,VaisalaTech,2024-11-16,21.2,65.4,5.8,6.8,346.0
1,MTO-0001,46.034407,0.042128,832,MeteoSens,2024-04-30,21.1,63.6,0.0,36.9,204.0
2,MTO-0002,46.277010,-0.954927,97,VaisalaTech,2024-02-27,19.5,66.2,9.4,16.7,103.0
3,MTO-0003,45.459713,2.444795,673,MeteoSens,2024-10-26,21.9,77.1,0.0,16.3,149.0
4,MTO-0004,46.672341,4.466764,731,MeteoSens,2025-09-05,21.8,67.5,0.6,15.1,332.0


In [29]:
carte_capteurs_plotly(df_meteo, "latitude", "longitude", "temperature_c", "id_station_meteo", "Réseau météo — température (°C)",
                       colonnes_survol={"humidite_pct": True, "vent_vitesse_kmh": True})


### 3.2. Réseau qualité de l'air urbain

**Méthodologie.** Réseau très majoritairement positionné près des villes (95% des capteurs tirés avec biais urbain). Les concentrations suivent le champ d'influence urbaine défini en partie 1 :

- **NO2** (trafic routier) : fond rural ~8 µg/m³, jusqu'à +70 µg/m³ au cœur des grandes agglomérations ;
- **PM10** : fond rural ~10 µg/m³, jusqu'à +45 µg/m³ en zone urbaine dense ;
- **PM2.5** : dérivée de PM10 (~60% de sa valeur, les particules fines étant une sous-fraction des particules totales), avec son propre bruit.

Colonne utile mais non numérique : `type_implantation` (`fond urbain` / `trafic` / `industriel`) — une vraie campagne de mesure officielle distingue ces trois types de sites, et un capteur `trafic` ne mesure pas la même chose qu'un capteur `fond urbain` (c'est un vrai critère de filtrage, pas un piège). Colonne inutile : `nom_station`.


In [30]:
n = N_CAPTEURS["qualite_air_urbain"]
positions = tirer_positions(n, RNG, biais_villes=0.95)
lons, lats = zip(*positions)
influence = influence_urbaine(np.array(lats), np.array(lons))

no2_ugm3 = np.clip(8 + 70 * influence + RNG.normal(0, 4, n), 0, 250)
pm10_ugm3 = np.clip(10 + 45 * influence + RNG.normal(0, 4, n), 0, 200)
pm25_ugm3 = np.clip(0.6 * pm10_ugm3 + RNG.normal(0, 2, n), 0, 150)

df_aq_urbain = pd.DataFrame({
    "code_capteur_aq": [f"AQ-{i:04d}" for i in range(n)],
    "latitude": lats,
    "longitude": lons,
    "nom_station": [f"Station {i}" for i in range(n)],
    "type_implantation": RNG.choice(["fond urbain", "trafic", "industriel"], n, p=[0.6, 0.3, 0.1]),
    "no2_ugm3": no2_ugm3.round(1),
    "pm10_ugm3": pm10_ugm3.round(1),
    "pm25_ugm3": pm25_ugm3.round(1),
})

for col in ["no2_ugm3", "pm10_ugm3", "pm25_ugm3"]:
    df_aq_urbain[col] = injecter_nan_aberrant(df_aq_urbain[col], TAUX_NAN["qualite_air_urbain"], TAUX_ABERRANT["qualite_air_urbain"], RNG)

df_aq_urbain.head()


,code_capteur_aq,latitude,longitude,nom_station,type_implantation,no2_ugm3,pm10_ugm3,pm25_ugm3
0,AQ-0000,48.978319,1.688577,Station 0,industriel,34.6,27.2,16.1
1,AQ-0001,45.953562,4.658458,Station 1,trafic,114.3,16.4,10.9
2,AQ-0002,49.106580,0.534900,Station 2,fond urbain,5.6,2.5,7.3
3,AQ-0003,45.389161,4.583171,Station 3,trafic,14.8,7.1,4.6
4,AQ-0004,48.480371,7.515814,Station 4,fond urbain,6.5,16.5,10.6


In [31]:
carte_capteurs_plotly(df_aq_urbain, "latitude", "longitude", "no2_ugm3", "code_capteur_aq", "Qualité de l'air urbain — NO2 (µg/m³)",
                       colonnes_survol={"type_implantation": True, "pm10_ugm3": True})


### 3.3. Réseau ozone rural

**Méthodologie.** Contrairement aux autres polluants, l'ozone (O3) n'augmente pas près des villes : le NO2 du trafic le « consomme » chimiquement (titration), si bien que l'ozone est souvent *plus* concentré en périphérie/zone rurale qu'en centre-ville. Les réseaux de mesure officiels placent d'ailleurs volontairement leurs stations de « fond rural » loin des sources directes de pollution — on reproduit cette règle de positionnement (rejet des tirages dont l'influence urbaine dépasse 0.25).

- **O3** : fond ~55 µg/m³, légèrement plus élevé au sud (plus d'ensoleillement), légèrement réduit par la proximité urbaine (-18 µg/m³ au maximum).

Colonnes inutiles : `gestionnaire_reseau`, `numero_serie`.


In [32]:
n = N_CAPTEURS["ozone_rural"]
positions = tirer_positions(n, RNG, biais_villes=0.0, eviter_villes_seuil=0.25)
lons, lats = zip(*positions)
influence = influence_urbaine(np.array(lats), np.array(lons))

o3_ugm3 = np.clip(55 + (47 - np.array(lats)) * 3 - 18 * influence + RNG.normal(0, 6, n), 5, 180)

df_o3_rural = pd.DataFrame({
    "ref_station_o3": [f"O3R-{i:04d}" for i in range(n)],
    "latitude": lats,
    "longitude": lons,
    "gestionnaire_reseau": RNG.choice(["AtmoReseau", "AirSurveil", "QualitAir"], n),
    "numero_serie": RNG.integers(100000, 999999, n),
    "o3_ugm3": o3_ugm3.round(1),
})

df_o3_rural["o3_ugm3"] = injecter_nan_aberrant(df_o3_rural["o3_ugm3"], TAUX_NAN["ozone_rural"], TAUX_ABERRANT["ozone_rural"], RNG)
df_o3_rural.head()


,ref_station_o3,latitude,longitude,gestionnaire_reseau,numero_serie,o3_ugm3
0,O3R-0000,48.211966,5.644248,AtmoReseau,449449,51.5
1,O3R-0001,50.249758,2.273590,AirSurveil,809611,46.0
2,O3R-0002,45.123563,-0.610645,AirSurveil,989303,65.1
3,O3R-0003,48.970265,5.765866,AtmoReseau,923592,NaN
4,O3R-0004,48.403254,-1.046246,AtmoReseau,106295,54.7


In [33]:
carte_capteurs_plotly(df_o3_rural, "latitude", "longitude", "o3_ugm3", "ref_station_o3", "Ozone rural — O3 (µg/m³)")


### 3.4. Réseau citoyen (capteurs low-cost PM)

**Méthodologie.** Réseau communautaire type « capteurs connectés » : très dense, très majoritairement urbain (90% biais ville), mais nettement plus bruité que le réseau officiel — taux de NaN et de valeurs aberrantes les plus élevés de toute la base. On reproduit aussi un biais documenté dans la littérature sur les capteurs optiques bon marché : ils ont tendance à **surestimer** les PM10 par rapport aux stations de référence (+15% en moyenne ici).

Colonnes inutiles : `modele_capteur`, `proprietaire_pseudo`.


In [34]:
n = N_CAPTEURS["citoyen_pm"]
positions = tirer_positions(n, RNG, biais_villes=0.90)
lons, lats = zip(*positions)
influence = influence_urbaine(np.array(lats), np.array(lons))

pm10_ugm3 = np.clip((10 + 45 * influence) * 1.15 + RNG.normal(0, 8, n), 0, 250)

df_citoyen_pm = pd.DataFrame({
    "capteur_uid_pm": [f"PMC-{i:05d}" for i in range(n)],
    "latitude": lats,
    "longitude": lons,
    "modele_capteur": RNG.choice(["SensoBox v2", "AirPi", "CleanDot"], n),
    "proprietaire_pseudo": [f"user_{j:05d}" for j in RNG.integers(0, 99999, n)],
    "pm10_ugm3": pm10_ugm3.round(1),
})

df_citoyen_pm["pm10_ugm3"] = injecter_nan_aberrant(df_citoyen_pm["pm10_ugm3"], TAUX_NAN["citoyen_pm"], TAUX_ABERRANT["citoyen_pm"], RNG)
df_citoyen_pm.head()


,capteur_uid_pm,latitude,longitude,modele_capteur,proprietaire_pseudo,pm10_ugm3
0,PMC-00000,46.071694,4.798674,AirPi,user_64332,9.6
1,PMC-00001,43.773830,1.856857,SensoBox v2,user_64283,22.6
2,PMC-00002,48.844163,2.914135,SensoBox v2,user_45872,24.9
3,PMC-00003,48.297776,7.507009,AirPi,user_96545,18.2
4,PMC-00004,43.512525,4.980052,CleanDot,user_85043,21.5


In [35]:
carte_capteurs_plotly(df_citoyen_pm, "latitude", "longitude", "pm10_ugm3", "capteur_uid_pm", "Capteurs citoyens — PM10 (µg/m³)")


## 4. Export en base DuckDB

On écrit les quatre tables dans un seul fichier `data/qualite_air.duckdb`. C'est ce fichier que vous ouvrirez dans `exercice.ipynb` avec `duckdb.connect(...)`, et que vous pourrez aussi explorer avec un logiciel de visualisation de base de données si vous le souhaitez.


In [36]:
CHEMIN_DB.unlink(missing_ok=True)
con = duckdb.connect(str(CHEMIN_DB))

con.execute("CREATE TABLE releves_meteo AS SELECT * FROM df_meteo")
con.execute("CREATE TABLE releves_qualite_air_urbain AS SELECT * FROM df_aq_urbain")
con.execute("CREATE TABLE releves_ozone_rural AS SELECT * FROM df_o3_rural")
con.execute("CREATE TABLE releves_citoyen_pm AS SELECT * FROM df_citoyen_pm")

print(con.sql("SHOW TABLES"))
for table in ["releves_meteo", "releves_qualite_air_urbain", "releves_ozone_rural", "releves_citoyen_pm"]:
    print(table, "->", con.sql(f"SELECT COUNT(*) FROM {table}").fetchone()[0], "lignes")

con.close()


┌────────────────────────────┐
│            name            │
│          varchar           │
├────────────────────────────┤
│ releves_citoyen_pm         │
│ releves_meteo              │
│ releves_ozone_rural        │
│ releves_qualite_air_urbain │
└────────────────────────────┘

releves_meteo -> 180 lignes
releves_qualite_air_urbain -> 220 lignes
releves_ozone_rural -> 90 lignes
releves_citoyen_pm -> 500 lignes


## 5. La formule de qualité de l'air (méthodologie, à implémenter en TP)

Dans `exercice.ipynb`, vous allez devoir construire vous-mêmes une fonction qui, à partir des relevés de capteurs, calcule un **indice de qualité de l'air à 5 niveaux** : Très bon, Bon, Moyen, Dégradé, Très dégradé. Voici la méthode (volontairement décrite ici en mots et en formules, pas en code Python : c'est votre travail de la traduire).

**Étape 1 — un sous-indice par polluant.** Comme les indices officiels (ATMO en France, CAQI en Europe), chaque polluant est converti en un niveau 1 (Très bon) à 5 (Très dégradé) via des seuils de concentration. Des seuils réalistes (en µg/m³) :

| Niveau | PM10 | PM2.5 | NO2 | O3 |
|---|---|---|---|---|
| 1 — Très bon | ≤ 20 | ≤ 10 | ≤ 40 | ≤ 50 |
| 2 — Bon | ≤ 40 | ≤ 20 | ≤ 90 | ≤ 100 |
| 3 — Moyen | ≤ 50 | ≤ 25 | ≤ 120 | ≤ 130 |
| 4 — Dégradé | ≤ 80 | ≤ 50 | ≤ 230 | ≤ 180 |
| 5 — Très dégradé | > 80 | > 50 | > 230 | > 180 |

**Étape 2 — le pire polluant gagne.** L'indice global avant correction météo est le **maximum** des 4 sous-indices (c'est la règle utilisée par les vrais indices : un seul polluant très dégradé suffit à dégrader tout l'indice, même si les trois autres sont excellents).

**Étape 3 — modulation météo.** Le vent disperse ou concentre la pollution :
- si le vent est très faible (< 5 km/h), la pollution stagne : on dégrade l'indice d'un niveau (sans dépasser 5) ;
- si le vent est fort (> 25 km/h), la dispersion est bonne : on peut améliorer l'indice d'un niveau (sans descendre sous 1).

**Exemples** (à titre d'illustration — vos résultats sur la vraie base peuvent différer) :

| PM10 | PM2.5 | NO2 | O3 | Vent | Sous-indices | Pire | Modulation | Indice final |
|---|---|---|---|---|---|---|---|---|
| 15 | 8 | 25 | 60 | 18 km/h | 1,1,1,2 | 2 | aucune | **Bon** |
| 55 | 30 | 70 | 90 | 3 km/h | 4,4,2,2 | 4 | vent faible → +1 | **Très dégradé** |
| 90 | 60 | 40 | 70 | 30 km/h | 5,5,1,2 | 5 | vent fort → -1 | **Dégradé** |

**Bonus — un score de confiance.** Au point `y` où vous interpolez une variable, la formule du TP donne `f_j(y) = Σ w_i f_j(x_i) / Σ w_i`. Le dénominateur `Σ w_i` est en lui-même une information : il est grand quand il y a beaucoup de capteurs proches, petit quand `y` est loin de tout. C'est une base naturelle pour un indice de confiance — à vous d'explorer sa distribution (histogramme) pour décider à partir de quel seuil une estimation doit être considérée comme peu fiable (et basculer en « donnée non disponible » plutôt que d'afficher un chiffre trompeur).


## Et maintenant ?

Ouvrez `exercice.ipynb` : vous y retrouverez `data/qualite_air.duckdb` avec ses 4 tables, et un fil d'étapes guidées pour agréger, nettoyer, construire l'indice, mailler la France et l'afficher sur une carte interactive.
